In [5]:
import time
import pandas as pd

class ECLAT_Optimized:
    def __init__(self, country_name, min_support_pct=0.05, max_length=3):
        """
        Initializes the ECLAT algorithm.
        :param min_support_pct: Minimum support threshold (e.g., 0.05 for 5%).
        :param max_length: Maximum number of items in a set (optimization to prevent long runtimes).
        """
        self.country = country_name
        self.min_support_pct = min_support_pct
        self.max_length = max_length
        self.frequent_itemsets = {}
        
    def transform_data(self, df):
        """
        Converts horizontal data (Transaction -> Item) to vertical (Item -> Set of TIDs).
        """
        return df.groupby('Item')['Transaction_ID'].apply(set).to_dict()

    def run(self, df):
        print(f"--- Running Optimized ECLAT for {self.country} ---")
        vertical_db = self.transform_data(df)
        
        total_trans = df['Transaction_ID'].nunique()
        min_sup_count = int(total_trans * self.min_support_pct)
        print(f"Transactions: {total_trans} | Min Support: {min_sup_count} ({self.min_support_pct*100}%)")
        
        # Filter initial items by support
        frequent_single_items = {
            k: v for k, v in vertical_db.items() 
            if len(v) >= min_sup_count
        }
        
        # Sort by frequency (Critical optimization for intersection speed)
        sorted_items = sorted(
            frequent_single_items.keys(), 
            key=lambda k: len(frequent_single_items[k]), 
            reverse=True
        )
        
        start_time = time.time()

        def _recursive_eclat(prefix, current_tids, start_index):
            # Stop if max_length is reached
            if len(prefix) >= self.max_length:
                return

            for i in range(start_index, len(sorted_items)):
                item = sorted_items[i]
                
                # INTERSECTION: The core ECLAT operation
                new_tids = current_tids.intersection(vertical_db[item])
                support = len(new_tids)
                
                if support >= min_sup_count:
                    new_itemset = tuple(sorted(prefix + [item]))
                    self.frequent_itemsets[new_itemset] = support
                    
                    # Recursion
                    _recursive_eclat(list(new_itemset), new_tids, i + 1)

        # Start the Depth-First Search
        for i, item in enumerate(sorted_items):
            tids = vertical_db[item]
            self.frequent_itemsets[(item,)] = len(tids)
            _recursive_eclat([item], tids, i + 1)
            
        exec_time = time.time() - start_time
        print(f"Done! Found {len(self.frequent_itemsets)} sets in {exec_time:.4f} seconds.")

    def get_top_rules(self, top_n=10):
        """
        Returns top frequent itemsets with more than 1 item.
        """
        # Filter for combinations only (length > 1)
        combos = {k: v for k, v in self.frequent_itemsets.items() if len(k) > 1}
        return sorted(combos.items(), key=lambda x: x[1], reverse=True)[:top_n]

In [8]:
# --- EXECUTION FOR ROMANIA ---
try:
    # 1. Load the clean data generated in the previous step
    # Ensure 'Romania_foodex_clean.csv' exists in your folder
    df_ro_clean = pd.read_csv('Romania_foodex_clean.csv')
    
    # 2. Run ECLAT
    # We use 0.05 (5%) support and max_length=3 for speed
    model_ro = ECLAT_Optimized(country_name="Romania", min_support_pct=0.02, max_length=3)
    model_ro.run(df_ro_clean)
    
    # 3. Display Results
    print("\n--- Top Frequent Itemsets (Romania) ---")
    top_rules = model_ro.get_top_rules(15)
    
    if not top_rules:
        print("No combinations found. Try lowering min_support_pct.")
    else:
        for items, supp in top_rules:
            print(f"{items}: {supp}")

except FileNotFoundError:
    print("Error: 'Romania_foodex_clean.csv' not found. Please run the cleaning function first.")

--- Running Optimized ECLAT for Romania ---
Transactions: 9664 | Min Support: 193 (2.0%)
Done! Found 15312 sets in 2.6209 seconds.

--- Top Frequent Itemsets (Romania) ---
('salt', 'sunflower seed oil'): 8298
('onions', 'salt'): 8015
('salt', 'wheat bread and rolls'): 7718
('carrots', 'salt'): 7479
('onions', 'sunflower seed oil'): 7416
('onions', 'salt', 'sunflower seed oil'): 7366
('carrots', 'onions'): 7326
('carrots', 'onions', 'salt'): 7284
('sunflower seed oil', 'wheat bread and rolls'): 7115
('salt', 'sunflower seed oil', 'wheat bread and rolls'): 7033
('onions', 'wheat bread and rolls'): 6964
('onions', 'salt', 'wheat bread and rolls'): 6905
('carrots', 'sunflower seed oil'): 6844
('carrots', 'salt', 'sunflower seed oil'): 6804
('carrots', 'onions', 'sunflower seed oil'): 6704


In [10]:
# --- EXECUTION FOR NIGERIA ---
try:
   
    df_ng_clean = pd.read_csv('Nigeria_foodex_clean.csv')
    
    # 2. Run ECLAT
    model_ng = ECLAT_Optimized(country_name="Nigeria", min_support_pct=0.02, max_length=3)
    model_ng.run(df_ng_clean)
    
    # 3. Display Results
    print("\n--- Top Frequent Itemsets (Nigeria) ---")
    top_rules = model_ng.get_top_rules(15)
    
    if not top_rules:
        print("No combinations found. Try lowering min_support_pct to.")
    else:
        for items, supp in top_rules:
            print(f"{items}: {supp}")

except FileNotFoundError:
    print("Error: 'Nigeria_foodex_clean.csv' not found. Please run the cleaning function first.")

--- Running Optimized ECLAT for Nigeria ---
Transactions: 990 | Min Support: 19 (2.0%)
Done! Found 355 sets in 0.0100 seconds.

--- Top Frequent Itemsets (Nigeria) ---
('mammals and birds meat', 'rice grain'): 300
('rice grain', 'soups (ready-to-eat)'): 292
('mammals and birds meat', 'soups (ready-to-eat)'): 288
('cassava roots', 'soups (ready-to-eat)'): 208
('soups (ready-to-eat)', 'wheat bread and rolls'): 165
('mammals and birds meat', 'rice grain', 'soups (ready-to-eat)'): 155
('fish (meat)', 'rice grain'): 155
('dried starchy roots and tuber products', 'soups (ready-to-eat)'): 154
('soups (ready-to-eat)', 'yams'): 153
('mammals and birds meat', 'wheat bread and rolls'): 152
('fish (meat)', 'soups (ready-to-eat)'): 145
('rice grain', 'wheat bread and rolls'): 140
('beans (dry) and similar-', 'soups (ready-to-eat)'): 136
('cassava roots', 'rice grain'): 119
('cocoa beverage-preparation', 'wheat bread and rolls'): 115
